In [ ]:
import json
import os
from pathlib import Path

from openai import OpenAI

# playground 로 옮긴 뒤에도 키가 잡히게
HERE = Path.cwd()
if (HERE / "llm-api-playground").is_dir():
    HERE = HERE / "llm-api-playground"
for env_path in (HERE / ".env", HERE / "env"):
    if env_path.exists():
        for line in env_path.read_text(encoding="utf-8").splitlines():
            if "=" in line and "key" in line.lower():
                os.environ["OPENAI_API_KEY"] = line.split("=", 1)[1].strip()

API_MODEL = "gpt-5.4-nano"
client = OpenAI()
print("키 로드:", "OK" if os.environ.get("OPENAI_API_KEY") else "실패 — env/.env 확인")

In [ ]:
r = client.chat.completions.create(
    model=API_MODEL,
    messages=[
        {"role": "user", "content": "오늘 부산 날씨 알려줘"}
    ],
    max_completion_tokens=500,
)
r.choices[0].message.content

In [ ]:
import urllib.request

def get_weather(city: str, latitude: float, longitude: float) -> str:
    """그 좌표의 지금 날씨를 open-meteo 에서 가져온다 (키 불필요)"""
    url = (
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}"
        f"&current=temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m&timezone=auto"
    )
    with urllib.request.urlopen(url, timeout=10) as resp:
        c = json.load(resp)["current"]
    return (
        f"{city} 기온 {c['temperature_2m']}도, 습도 {c['relative_humidity_2m']}%, "
        f"강수 {c['precipitation']}mm, 바람 {c['wind_speed_10m']}m/s"
    )

tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "그 도시의 지금 날씨를 알려 준다. 위도·경도는 네가 아는 값을 채워라.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "도시 이름"},
                "latitude": {"type": "number", "description": "그 도시의 위도"},
                "longitude": {"type": "number", "description": "그 도시의 경도"},
            },
            "required": ["city", "latitude", "longitude"],
            "additionalProperties": False,
        },
    },
}]
print("함수 준비 완료 — 아직 아무도 부르지 않았다")

In [ ]:
messages = [{"role": "user", "content": "오늘 부산 날씨 알려줘"}]
r = client.chat.completions.create(
    model=API_MODEL,
    messages=messages,
    tools=tools,
    max_completion_tokens=500,
)
print("finish_reason:", r.choices[0].finish_reason)
r.choices[0].message

In [ ]:
msg = r.choices[0].message
tc = msg.tool_calls[0]  # 함수 호출
# 모델이 자기 일을 종료하고, 파이썬에게 도구 호출을 맡긴다.
print(tc.function.name)
print(tc.function.arguments)
args = json.loads(tc.function.arguments)
print(args)
weather = get_weather(**args)
print(weather)

In [ ]:
messages.append(msg)
messages.append({
    "role": "tool",
    "tool_call_id": tc.id,
    "content": weather,
})
final = client.chat.completions.create(
    model=API_MODEL,
    messages=messages,
    tools=tools,
    max_completion_tokens=500,
)
print(final.choices[0].finish_reason)
print(final.choices[0].message.content)

In [ ]:
import re

def calc(expression: str) -> str:
    """수식 문자열을 계산한다. 예: '739521468 * 8462137'"""
    if not re.fullmatch(r"[0-9+\-*/(). %]+", expression) or "**" in expression or len(expression) > 80:
        return "허용되지 않는 수식이라 계산을 거부했다"
    return str(eval(expression, {"__builtins__": {}}, {}))

tools.append({
    "type": "function",
    "function": {
        "name": "calc",
        "description": "수식을 정확하게 계산한다. 예: 739521468 * 8462137",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "계산할 수식"}
            },
            "required": ["expression"],
            "additionalProperties": False,
        },
    },
})
print("calc 도구 추가됨, tools 개수:", len(tools))

In [ ]:
FUNCTIONS = {"get_weather": get_weather, "calc": calc}
MAX_ROUNDS = 5


def run_with_tools(user_text: str) -> str:
    history = [{"role": "user", "content": user_text}]
    for _ in range(MAX_ROUNDS):
        resp = client.chat.completions.create(
            model=API_MODEL,
            messages=history,
            tools=tools,
        )
        out = resp.choices[0].message
        if resp.choices[0].finish_reason != "tool_calls":
            return out.content or "(빈 응답)"
        history.append(out)
        for call in out.tool_calls:
            fn = FUNCTIONS.get(call.function.name)
            call_args = json.loads(call.function.arguments)
            result = fn(**call_args) if fn else f"알 수 없는 함수: {call.function.name}"
            history.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": result if isinstance(result, str) else json.dumps(result, ensure_ascii=False),
            })
    return "(도구 호출 상한 도달)"


print(run_with_tools("부산 날씨 알려주고, 27 * 34 도 계산해줘"))